# Audit-only cache for ASL dataset

Chạy notebook này trên Google Colab qua `colab exec`. Nó chỉ tải raw archive, kiểm tra ảnh, tính SHA-256, phát hiện duplicate và tạo metadata tái sử dụng. Nó không chạy MediaPipe hoặc train model.


In [ ]:
%pip -q install 'huggingface-hub>=0.25' pandas pyarrow opencv-python-headless


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, zipfile
import cv2, pandas as pd
from huggingface_hub import snapshot_download

REPO_ID = 'hnam25/asl-hand-gesture-images'
REVISION = 'bad9dd9297697ec2da901562b203a5a14ebb93d1'
RAW_ARCHIVE = 'ASL_HG_36000/ASL_Raw_Images.zip'
AUDIT_VERSION = 'colab-audit-2026-08-10'
ROOT = Path('/content/asl-audit')
RAW, METADATA = ROOT / 'raw', ROOT / 'metadata' / AUDIT_VERSION
RAW.mkdir(parents=True, exist_ok=True); METADATA.mkdir(parents=True, exist_ok=True)
CLASSES = [str(i) for i in range(10)] + [chr(i) for i in range(ord('A'), ord('Z') + 1)]
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


In [ ]:
snapshot_download(repo_id=REPO_ID, repo_type='dataset', revision=REVISION, local_dir=RAW, allow_patterns=[RAW_ARCHIVE])
archive_path = RAW / RAW_ARCHIVE
if not archive_path.is_file():
    raise RuntimeError(f'Missing {RAW_ARCHIVE} at revision {REVISION}')
raw_archive_sha256 = sha256_file(archive_path)
with zipfile.ZipFile(archive_path) as archive:
    for member in archive.infolist():
        destination = (RAW / member.filename).resolve()
        if RAW.resolve() not in destination.parents and destination != RAW.resolve():
            raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
    archive.extractall(RAW)
class_roots = [p.parent for p in RAW.rglob('0') if p.is_dir() and all((p.parent / label).is_dir() for label in CLASSES)]
if not class_roots:
    raise RuntimeError('Could not find direct 0-9/A-Z class folders after extraction.')
class_root = sorted(class_roots, key=lambda p: len(p.parts))[0]
print('Dataset revision:', REVISION)
print('Raw archive SHA-256:', raw_archive_sha256)
print('Class root:', class_root)


In [ ]:
rows = []
for label in CLASSES:
    class_dir = class_root / label
    if not class_dir.is_dir():
        rows.append({'label': label, 'relative_path': '', 'status': 'missing_class_directory', 'sha256': ''})
        continue
    for path in sorted(class_dir.rglob('*')):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        image = cv2.imread(str(path))
        rows.append({
            'label': label,
            'relative_path': path.relative_to(class_root).as_posix(),
            'status': 'ok' if image is not None else 'unreadable',
            'sha256': sha256_file(path) if image is not None else '',
        })
    if len(rows) % 1000 == 0:
        print(f'Audited {len(rows):,} files', flush=True)

audit = pd.DataFrame(rows)
readable = audit[audit.status == 'ok'].copy()
duplicates = readable[readable.sha256.duplicated(False)].sort_values(['sha256', 'relative_path'])
class_distribution = readable.groupby('label').size().rename('images').reindex(CLASSES, fill_value=0).reset_index()
audit.to_csv(METADATA / 'audit.csv', index=False)
audit.to_parquet(METADATA / 'audit.parquet', index=False)
duplicates.to_csv(METADATA / 'duplicates.csv', index=False)
duplicates.to_parquet(METADATA / 'duplicates.parquet', index=False)
class_distribution.to_csv(METADATA / 'class_distribution.csv', index=False)

manifest = {
    'schema_version': 1,
    'kind': 'audit-only',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'dataset_repo': REPO_ID,
    'dataset_revision': REVISION,
    'raw_archive': RAW_ARCHIVE,
    'raw_archive_sha256': raw_archive_sha256,
    'classes': CLASSES,
    'image_extensions': sorted(IMAGE_EXTENSIONS),
    'totals': {
        'records': int(len(audit)),
        'readable': int((audit.status == 'ok').sum()),
        'unreadable': int((audit.status == 'unreadable').sum()),
        'duplicate_rows': int(len(duplicates)),
        'duplicate_hash_groups': int(duplicates.sha256.nunique()),
    },
}
(METADATA / 'cache_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest['totals'], indent=2))
assert manifest['totals']['readable'] > 0
assert class_distribution.images.min() >= 10


## Download artifacts

Sau khi notebook hoàn tất, download toàn bộ thư mục `/content/asl-audit/metadata/colab-audit-2026-08-10/` về local. Xác minh `cache_manifest.json` trước khi upload directory này lên `metadata/colab-audit-2026-08-10/` của Hugging Face.
